# C10-competition-craft — Session 3: The Notebook and the Writeup

*One class session, roughly 85 minutes. Prerequisites: Sessions 1–2
(the contract, the harness, macro-F1, the iteration loop).*

**This session:** the last two graded surfaces.
First, **notebook discipline** — the four rules that make your
notebook survive a stranger's "Restart & Run All", plus the
determinism audit that proves it.
Second, **writeup quality** — the summary cell where you explain your
approach, your intuition, and the alternatives you weighed; on the
real exam this reasoning is *explicitly graded*, so we grade it here
too, with a rubric you will apply yourself.
The session (and the teaching arc of this course) closes with the full
worked mini-competition: every device from Sessions 1–3 assembled into
one clean submission.

Try every checkpoint yourself before reading on.
Answers are collected at the end of this notebook.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804

df = pd.read_csv("../data/train.csv")
FEATURES = [c for c in df.columns if c != "outcome"]
X = df[FEATURES]
y = df["outcome"].to_numpy()

## 1. Notebook Discipline: Four Rules

**Motivation.**
Session 1 said it: the grader runs your notebook top to bottom on a
fresh machine.
Everything you did out of order, everything living only in your
kernel's memory, everything random and unpinned — gone or changed.
Notebook discipline is the craft of making the *file* equal the
*work*.

**The four rules:**

| # | Rule | What it prevents |
|---|------|------------------|
| D1 | **Pin every seed.** One `SEED` constant at the top; every random operation (`default_rng`, `random_state=`) uses it or a value derived from it. | results that change between runs |
| D2 | **Cell order = execution order.** The notebook must produce its results when run 1, 2, 3, …, top to bottom, fresh kernel. | out-of-order dependencies that only your kernel remembers |
| D3 | **No dead cells.** Every cell either computes something used later, produces a reported result, or is deliberate narrative. Failed experiments live in the log (Session 2), not as commented-out rubble. | half-broken cells that crash the grader's run |
| D4 | **Deterministic re-run.** Two fresh top-to-bottom runs produce identical results — numbers, predictions, all of it. D1–D3 make it possible; the audit (Section 2) makes it *checked*. | "it worked when I ran it" |

**The test each rule implies** — before submitting, literally do:
Restart kernel → Run All → read every output → Restart → Run All
again → compare.
Ten minutes, and it catches the entire class of zeros that no amount
of modeling skill survives.

### Checkpoint 1

1. Which rule does each scenario break?
   (i) A cell defines `best_k` *below* the cell that uses it — fine
   in your session because you ran them out of order.
   (ii) A `train_test_split` without `random_state`.
   (iii) A cell reading `df_old`, a name from an experiment you
   deleted the cell for.
2. Why is D3 a *grading* concern and not just tidiness?
   (What does one crashing dead cell do to the run-clean component?)
3. D4 is listed as a consequence of D1–D3 plus an audit.
   Give an example of a notebook satisfying D1–D3 that still fails
   D4.

## 2. Seeds and the Determinism Audit

**What an unpinned seed does.**
Your validation split is a random carve; unpinned, every fresh run
carves differently, and every number downstream — `val_f1`, `best_k`,
your log — silently changes.
Watch two *different* pinned seeds stand in for what two runs of an
*unpinned* notebook would print:

In [ ]:
for rs in (1, 2):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=150,
                                      random_state=rs, stratify=y)
    p = Pipeline([("scaler", StandardScaler()),
                  ("knn", KNeighborsClassifier(n_neighbors=5))]).fit(Xa, ya)
    print(f"carve seed {rs}: val macro-F1 ="
          f" {f1_score(yb, p.predict(Xb), average='macro'):.4f}")
print("an unpinned notebook prints a DIFFERENT one of these each run")

Two runs, two scores, one notebook — a grader re-running you gets
numbers your writeup never claimed.
With `random_state=SEED` everywhere, both runs print the same thing,
and your claims are checkable.

**The determinism audit** (D4, mechanized).
Wrap the entire pipeline — carve, fit, evaluate, predict — in one
function of no arguments, call it twice, and demand *exact* equality.
Not `isclose`: the same code on the same data with the same seeds has
no excuse for even the last bit to differ:

In [ ]:
def run_submission():
    # the WHOLE flow, from the raw table to the deliverables
    X_tr, X_val, y_tr, y_val = train_test_split(
        X, y, test_size=150, random_state=SEED, stratify=y)
    pipe = Pipeline([("scaler", StandardScaler()),
                     ("knn", KNeighborsClassifier(n_neighbors=11))])
    pipe.fit(X_tr, y_tr)
    val_f1 = f1_score(y_val, pipe.predict(X_val), average="macro")
    pipe.fit(X, y)                      # final refit on all rows
    probe_preds = pipe.predict(X.iloc[200:230])
    return val_f1, probe_preds


f1_a, preds_a = run_submission()
f1_b, preds_b = run_submission()
print("run A val_f1:", round(f1_a, 6))
print("run B val_f1:", round(f1_b, 6))
print("exactly equal:", f1_a == f1_b and (preds_a == preds_b).all())

`exactly equal: True` — the audit passes, and it would have caught an
unpinned seed, an accidental dependence on run order, or a stale
global, because any of those breaks bit-level equality somewhere.

### Checkpoint 2

1. Why does the audit demand `==` rather than `np.isclose`?
   When *would* a legitimate reproducibility check need a tolerance?
   (Think: same code, different machine/library versions.)
2. The audit function refits everything from scratch.
   Explain how that specifically catches Pitfall "stale globals"
   from Session 1.
3. A notebook's audit passes, but its printed `val_f1` differs from
   what `run_submission()` returns.
   What does that tell you about the notebook's cells?

## 3. The Writeup: Approach, Intuition, Alternatives

**Motivation.**
The exam's applied problem grades three things: the run, the
hidden-test score, and **the quality of your reasoning** — stated in
the rules, worth real points, and the part strong modelers most often
throw away by writing two vague sentences at 11:58pm.

**The deliverable** is one markdown cell, at the end of the
submission, with three labeled parts:

- **Approach** — *what you did*, precisely enough to reproduce
  without reading your code: data prep, model family and
  hyperparameter values, feature set, validation protocol, and the
  validation score of the submitted model.
- **Intuition** — *why it should work*: the property of the data
  that makes the model fit, and the property of the task that makes
  the metric right.
- **Alternatives** — *what else you weighed*: at least one concrete
  alternative with its measured outcome (your iteration log is the
  source), and an honest limitation or next step.

**The rubric** — this course's grading standard for that cell, six
points, every criterion mechanically checkable:

| Part | +1 if the writeup… | +1 more if it… |
|------|---------------------|----------------|
| W-A Approach | names the full recipe: preprocessing, model family **with hyperparameter values**, feature set | states the validation protocol (carve size, seed policy, stratification) **and** the final model's validation metric value |
| W-B Intuition | grounds the model choice in a property of *this data* (not "kNN is good") | grounds the metric choice in a property of *this task* (imbalance → macro-F1) |
| W-C Alternatives | names ≥1 concrete alternative **with its measured outcome** or precise reason for rejection | states an honest limitation or next step (e.g. validation-optimism caveat, untried idea) |

Grade: 6/6 is normal, attainable craft — every point is a sentence
you already have the facts for by the time you finish iterating.

### Checkpoint 3

1. Why does W-A's second point demand the validation *protocol* and
   not just the score?
   (What claim is uncheckable without it?)
2. "We chose kNN because it is a powerful algorithm" — which rubric
   point does this fail to earn and what one edit earns it?
3. Where do W-C's facts come from, mechanically, in a disciplined
   submission?

## 4. The Rubric, Applied

Two submitted writeups for the apiary task.
Score them yourself before reading the verdicts.

**Writeup 1:**

> We used machine learning to predict the bee colonies.
> We tried many models and settings and the results were very good.
> We scaled the features because kNN works on distances and the
> columns have very different scales.
> kNN was the best choice for this problem.

**Writeup 2:**

> **Approach.** Scaled 11-NN (`StandardScaler` +
> `KNeighborsClassifier(n_neighbors=11)`, pipeline) on the 7
> signal-bearing features (dropping the 5 columns whose class-mean
> gap was <0.2 stds).
> Validation: stratified 150-row carve, `random_state=SEED`, frozen
> throughout; final model's validation macro-F1 = 0.82, then refit
> on all 600 rows for submission.
> **Intuition.** kNN needs a distance that means something: scaling
> stops the large-scale noise columns (elevation, water distance)
> from dominating, and dropping them entirely removes five dimensions
> of pure static.
> Macro-F1 fits the task because the classes are ~2:1 imbalanced and
> the minority (struggling colonies) is what the apiary cares about.
> **Alternatives.** k = 1..11 swept: k = 5 scored 0.77, k = 11 won at
> 0.81 on all features; feature-dropping then added ~0.01.
> Limitation: three accepted changes were selected on one validation
> split, so 0.82 is likely a bit optimistic for unseen colonies.

Scoring, mechanically against the table:

In [ ]:
rubric = ["W-A1 full recipe (model + hyperparams + features)",
          "W-A2 validation protocol + final val score",
          "W-B1 model choice grounded in the data",
          "W-B2 metric choice grounded in the task",
          "W-C1 concrete alternative with outcome",
          "W-C2 honest limitation / next step"]

writeup1 = [0, 0, 1, 0, 0, 0]
writeup2 = [1, 1, 1, 1, 1, 1]

for crit, s1, s2 in zip(rubric, writeup1, writeup2):
    print(f"{crit:<50} W1: {s1}  W2: {s2}")
print(f"{'TOTAL':<50} W1: {sum(writeup1)}  W2: {sum(writeup2)}")

The verdicts, argued:

- **Writeup 1 — 1/6.**
  Its single point is W-B1: "scaled because kNN works on distances
  and the columns have different scales" genuinely grounds a choice
  in this data.
  Everything else fails mechanically: no hyperparameter values, no
  feature set, no protocol, no score (W-A: 0); "best choice" with no
  task property (W-B2: 0); "tried many models" with no named
  alternative or number (W-C: 0).
  Note the sting: its *model* might be identical to Writeup 2's.
- **Writeup 2 — 6/6.**
  Every criterion is a checkable sentence: recipe with values,
  protocol with carve details, score, two grounded intuitions, a
  sweep with numbers, and the selection-optimism caveat straight
  from Session 2 §6.

### Checkpoint 4

1. Score this fragment as W-C: "We also tried k = 3 (macro-F1 0.78,
   worse than 0.81) and considered dropping `queen_age_years` but
   kept it since its class gap is 0.5 stds."
2. Writeup 1 claims "results were very good."
   Which rubric point *could* that sentence have earned, and what
   two facts must be added?
3. Why is W-C2 (the limitation) worth a point at all — what does its
   presence signal to a grader about the *rest* of the writeup?

## 5. The Full Worked Mini-Competition

Everything, assembled — the dress rehearsal for the exam's 50-point
problem.
The notebook below is the *shape* of a submission: numbered stages,
each one cell, nothing dead, nothing out of order.

**Stage 1 — data in, first look** (Session 1 §2 did the looking; here
we keep only what the submission needs):

In [ ]:
# Stage 1: load + bridge (D1: SEED already pinned at the top)
print("rows:", len(df), "| features:", len(FEATURES), "| classes:",
      {k: int(v) for k, v in df["outcome"].value_counts().items()})

In [ ]:
# Stage 2: frozen validation carve (Session 2 -- the only honest signal)
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=150, random_state=SEED, stratify=y)


def val_f1_of(feats, k):
    pipe = Pipeline([("scaler", StandardScaler()),
                     ("knn", KNeighborsClassifier(n_neighbors=k))])
    pipe.fit(X_tr[feats], y_tr)
    return f1_score(y_val, pipe.predict(X_val[feats]), average="macro")

In [ ]:
# Stage 3: baseline + capped iteration (Session 2's campaign, replayed)
SIGNAL = ["honey_stores_kg", "autumn_hive_mass_kg", "varroa_mite_index",
          "forager_traffic_per_min", "brood_frames", "daily_temp_swing_c",
          "queen_age_years"]
ks = np.array([1, 3, 5, 7, 9, 11])
sweep = np.array([val_f1_of(FEATURES, int(k)) for k in ks])
best_k = int(ks[np.argmax(sweep)])

log = pd.DataFrame([
    {"step": "baseline", "change": "scaled 5-NN, all 12 features",
     "val_f1": val_f1_of(FEATURES, 5)},
    {"step": "iter-1", "change": f"k swept {[int(k) for k in ks]} -> {best_k}",
     "val_f1": sweep.max()},
    {"step": "iter-2", "change": "12 features -> 7 SIGNAL columns",
     "val_f1": val_f1_of(SIGNAL, best_k)},
])
print(log.to_string(index=False, formatters={"val_f1": "{:.4f}".format}))
final_val_f1 = float(log["val_f1"].iloc[-1])

In [ ]:
# Stage 4: FINAL MODEL -- refit the accepted recipe on ALL labeled rows.
# No cell below this one rebinds `final_pipe` or `predict_labels` (D2/D3).
final_pipe = Pipeline([("scaler", StandardScaler()),
                       ("knn", KNeighborsClassifier(n_neighbors=best_k))])
final_pipe.fit(X[SIGNAL], y)


def predict_labels(X_test):
    # Contract: any-length DataFrame with the training feature columns in
    # the training order; Series out, X_test's index, training vocabulary.
    return pd.Series(final_pipe.predict(X_test[SIGNAL]), index=X_test.index)


print("final model: scaled", best_k, "-NN on", len(SIGNAL), "features",
      f"| val macro-F1 {final_val_f1:.4f}")

In [ ]:
# Stage 5: contract self-checks on a mid-table probe (Session 1's habit)
probe = X.iloc[300:340]
out = predict_labels(probe)
print("R3 Series :", isinstance(out, pd.Series))
print("R4 length :", len(out) == len(probe))
print("R4 index  :", out.index.equals(probe.index))
print("R5 vocab  :", set(out.unique()) <= set(np.unique(y)))

In [ ]:
# Stage 6: determinism audit (Section 2) -- the whole flow, twice
def run_submission():
    X_a, X_b, y_a, y_b = train_test_split(
        X, y, test_size=150, random_state=SEED, stratify=y)
    p = Pipeline([("scaler", StandardScaler()),
                  ("knn", KNeighborsClassifier(n_neighbors=best_k))])
    p.fit(X_a[SIGNAL], y_a)
    vf1 = f1_score(y_b, p.predict(X_b[SIGNAL]), average="macro")
    p.fit(X[SIGNAL], y)
    return vf1, p.predict(X.iloc[300:340][SIGNAL])


(f1_a, pr_a), (f1_b, pr_b) = run_submission(), run_submission()
print("audit -- two fresh runs identical:",
      f1_a == f1_b and (pr_a == pr_b).all())

**Stage 7 — the writeup cell.**
In a real submission the following is a markdown cell (it is Writeup
2 from Section 4, which the rubric scored 6/6 — note that every
number in it appears in an output above):

> **Approach.** Scaled 11-NN (`StandardScaler` +
> `KNeighborsClassifier(n_neighbors=11)`, pipeline) on the 7
> signal-bearing features (dropping the 5 columns whose class-mean
> gap was <0.2 stds).
> Validation: stratified 150-row carve, `random_state=SEED`, frozen
> throughout; final model's validation macro-F1 = 0.82, then refit
> on all 600 rows for submission.
> **Intuition.** kNN needs a distance that means something: scaling
> stops the large-scale noise columns from dominating, and dropping
> them entirely removes five dimensions of pure static.
> Macro-F1 fits the task because the classes are ~2:1 imbalanced and
> the minority (struggling colonies) is what the apiary cares about.
> **Alternatives.** k = 1..11 swept: k = 5 scored 0.77, k = 11 won
> at 0.81 on all features; feature-dropping then added ~0.01.
> Limitation: three accepted changes were selected on one validation
> split, so 0.82 is likely a bit optimistic for unseen colonies.

And that is a complete submission: seven stages, one contract, one
honest number, one accountable story.
What the held-back macro-F1 will be, we — by protocol — do not know;
Session 2 §6 says only to expect it BELOW the last validation number —
possibly well below: THREE selection decisions were made on that one
carve, and Session 2's own seed experiment showed a 0.0567 spread from
carve luck alone. A drop that erases an accepted improvement is a live
possibility, not a tail risk. The writeup
says so out loud.

### Checkpoint 5

1. Map each of the seven stages to the session (1, 2, or 3) that
   taught its device.
2. Which stages may be reordered without breaking anything, and which
   are order-critical?
   (Check against D2 — what does each stage consume?)
3. The log shows iter-2's gain was ~0.01 — inside Session 2's
   weather zone.
   Make the honest argument for accepting it anyway, and the honest
   argument for rejecting it; what would each choice change in the
   writeup?

## Exam Connections

How this unit's material shows up in Round 1 (paraphrased from the
`reference/analysis.md` topic table — no verbatim test text):

- **The applied tabular-ML problem is the paper's single largest
  item** — a Kaggle-style notebook submission worth ~50 of 300
  points — and its printed grading criteria are this unit's three
  sessions verbatim: the notebook **runs start to finish**
  (Session 3's discipline), performance is measured **on a hidden
  test set** (Session 1's protocol, Session 2's metric), and
  **reasoning quality is graded** while code style explicitly is
  not (Session 3's writeup).
- The rules **restrict the model family to k-nearest neighbors with
  scikit-learn allowed** — C4's toolkit under C10's constraint
  register, practiced here with the same zero-points ban clauses the
  paper prints.
- The grading metric is a **macro-averaged F1**, exactly Session 2's
  scoreboard, on an imbalanced task where it genuinely diverges from
  accuracy.
- Deliverables are **named identifiers with pinned forms** — the
  paper's register everywhere — and this unit's
  `predict_labels`-style contract is that register applied to a
  prediction function.
- Concept MCs elsewhere on the paper probe the *protocol* facts:
  what leaks, what validation estimates, why test rows are touched
  once — Sessions 1–2's checkpoint material.

Highest-yield hour if the exam were tomorrow: rebuild the
mini-competition (Section 5) from a blank notebook against the clock,
then re-run its determinism audit and score your own writeup against
the rubric.

## Going Deeper

This is the sixteenth and final teaching unit of the syllabus — the
DAG ends here, and the forward pointer is no longer another unit:

- **`mocktests/`** — the course's mock-test track: full Round-1-shaped
  papers (blueprint-conformant problem mixes, point budgets, and time
  limits) assembled from everything the sixteen units taught.
  This unit is the bridge: a mock's applied problem is exactly a
  C10 mini-competition under exam conditions.
  Take them timed, notebook discipline and all — the grading
  harness runs your `predict_labels` the same way this unit's does.
- **`review.ipynb`** (this unit) — the consolidation pass: the
  contract clauses, the discipline rules, and the rubric as
  checklists, plus the self-quiz.
  Recommended *before* your first mock.
- **Beyond the course:** the craft in this unit is the entry skill of
  real tabular competitions (Kaggle and friends), where the same
  loop — frozen validation, capped iteration, honest writeups —
  scales up with fancier models in the kNN slot.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. (i) D2 — the file's order lies about the dependency; a fresh
   top-to-bottom run hits `best_k` before it exists.
   (ii) D1 — the carve changes every run.
   (iii) D3 (a dead cell that happens to still run — until it
   doesn't) and, on a fresh kernel, D2: `df_old` no longer exists,
   so the cell crashes the grader's run.
2. The grader's first criterion is "runs clean, top to bottom"; one
   dead cell raising `NameError` fails the run for the whole
   notebook — the model never gets to speak.
3. Any notebook with an *unseeded* source of variation that D1's
   letter misses — e.g. iterating over a Python `set` of feature
   names (iteration order varies between interpreter runs) and
   letting the order affect a result, or depending on wall-clock
   time or file listing order.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. Same machine, same environment, same seeds: the computation is a
   pure function and equality should be exact — a tolerance would
   only paper over a real nondeterminism.
   Cross-machine or cross-version reproduction is where floating
   point may legitimately differ in the last bits, and a stated
   tolerance (atol, rtol=0 in this course's register) becomes the
   right tool.
2. The function rebuilds carve, fit, and evaluation from the raw
   table each call — it cannot see stale globals; if the notebook's
   printed numbers came from a stale interactive state, the audit's
   fresh numbers expose the mismatch.
3. Some cell between the audit's definition and the printed number
   did something the function doesn't (or vice versa) — the
   notebook's visible flow and its actual flow have diverged, which
   is exactly a D2/D3 violation waiting to zero the run.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. The score alone is an unverifiable brag: without carve size, seed
   policy, and stratification, a grader (or teammate) cannot re-run
   the protocol and check the claim — and cannot tell an honest
   estimate from split shopping.
2. W-B1 — it names the model but grounds nothing in the data.
   Edit: replace "powerful" with the data property: "because after
   scaling, colonies with similar sensor profiles cluster by
   outcome, which is exactly the neighborhood structure kNN votes
   over."
3. From the iteration log — every alternative W-C wants was a logged
   attempt with its measured `val_f1`; a disciplined submission
   writes W-C by quoting its own log.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. Both points: a concrete alternative with its measured outcome
   (k = 3, 0.78 vs 0.81 — W-C1) *and* a reasoned keep decision…
   but note W-C2 (limitation/next step) is a separate criterion —
   the fragment as given earns W-C1 only, unless the keep-decision
   sentence is judged a "precise reason for rejection", which it is
   (of the dropping idea): W-C1 earned once, not twice.
   Score: W-C1 = 1, W-C2 = 0.
2. W-A2 — with the metric named and its value: "validation macro-F1
   was 0.82 under a stratified, seeded 150-row carve."
3. It signals calibration: a writer who states what the number does
   *not* promise has understood Session 2 §6, so a grader can trust
   the other claims were made with the same care.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Stage 1: Session 1 (§2, the harness); Stage 2: Session 2 (§4,
   the carve); Stage 3: Session 2 (§5, the loop); Stage 4: Session 1
   (§7, final refit); Stage 5: Session 1 (§3, the contract);
   Stage 6: Session 3 (§2, the audit); Stage 7: Session 3 (§3–4,
   the writeup).
2. Order-critical: 1 → 2 → 3 → 4 (each consumes the previous —
   data, carve, accepted recipe), and 5–6 must follow 4 (they test
   the final artifacts).
   5 and 6 may swap with each other; 7 must be last-ish only because
   it quotes final numbers.
3. Accept: the change has a *mechanism* (five demonstrably
   near-zero-signal columns removed — Section 2's table), and
   mechanism-backed gains inside the noise band are still principled
   simplifications; the writeup keeps "added ~0.01" honest.
   Reject: by the weather rule alone, 0.01 on 150 rows is not
   evidence; the writeup would then say "feature-dropping did not
   measurably help; kept all 12 features for simplicity."
   Either is defensible *because the writeup discloses it* — what is
   not defensible is silence.

</details>